In [1]:
import pandas as pd
import os
import re
import glob
from collections import Counter
from IPython.display import FileLink, display

In [2]:

data_path = "/kaggle/input/datasets/vinayreddyyerram/updated-project-data"

all_files = glob.glob(os.path.join(data_path, "*.csv"))

print("Files found:", len(all_files))

df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)

print("Total Raw Rows:", df.shape)

Files found: 74
Total Raw Rows: (179337, 13)


In [3]:
df = df[['QueryText', 'KccAns', 'QueryType', 'StateName', 'Crop']].copy()

df.columns = ['question', 'answer', 'intent', 'state', 'crop']

print(df.head())

# =====================================================
# STEP 3 : BASIC CLEANING
# =====================================================

df = df.dropna(subset=['question', 'answer'])
df = df.reset_index(drop=True)

print("After dropna:", df.shape)


                                            question  \
0                      Farmer asked query on Weather   
1                Asked about Raitha Bele Parihara.\n   
2  Asked about Early stem borer management in Sug...   
3                    Asked about Plant protection \n   
4  Asked about PM KISAN Schemes status claim and ...   

                                              answer                intent  \
0  ನಿಮ್ಮ ತಾಲೂಕಿನಲ್ಲಿ ಶೇಕಡಾ 40% ಮಳೆ ನಿರೀಕ್ಷೆ ಇದ್ದು...               Weather   
1  Raitha Call Center Toll Free No: 1800 425 3553...    Government Schemes   
2  Given neccessary information.\nಅಗತ್ಯ ಮಾಹಿತಿ ನೀ...  \tPlant Protection\t   
3  ಹತ್ತಿಯಲ್ಲಿನ ಕಪ್ಪು ನುಶಿ ಬಗ್ಗೆ ಕೇಳಿದರು \nಥಿಯಾಮೆಥ...  \tPlant Protection\t   
4  --ಅಗತ್ಯ ಮಾಹಿತಿ ನೀಡಿದ್ದು, ಆರ್. ಎಸ್. ಕೆ. ಅಥವಾ ಕೃ...    Government Schemes   

       state                    crop  
0  KARNATAKA          Cotton (Kapas)  
1  KARNATAKA                  Others  
2  KARNATAKA  Sugarcane (Noble Cane)  
3  KARNATAKA          Cotton (Kapas)  

In [4]:
INDIC_OR_NON_ENGLISH_RE = re.compile(
    r'[\u0900-\u0DFF\u0400-\u04FF\u0600-\u06FF\u3040-\u30FF\u4E00-\u9FFF\uAC00-\uD7AF]'
)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def has_non_english_script(text):
    return bool(INDIC_OR_NON_ENGLISH_RE.search(str(text)))

def is_english(text):
    text = str(text)
    return bool(re.search(r'[a-zA-Z]', text)) and not has_non_english_script(text)

def has_repeat_words(text):
    return bool(re.search(r'\b(\w+)\s+\1\b', str(text).lower()))

def is_valid(row):
    q = str(row['question']).lower().strip()
    a = str(row['answer']).lower().strip()

    if q in ["none", "nan", ""] or a in ["none", "nan", ""]:
        return False

    bad = [
        "blank call",
        "irrelevant call",
        "incomplete call"
    ]

    if any(x in q for x in bad):
        return False

    if len(q) < 5 or len(a) < 5:
        return False

    return True

def clean_question(text):
    text = str(text).lower()

    text = re.sub(r'[\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    text = re.sub(
        r'(farmer asked query on|farmer asked about|others asked query on|others asked about|asked about|query on|query related to)',
        ' ',
        text
    )

    text = re.sub(r'\b(farmer|others|asked|query|about)\b', ' ', text)

    text = re.sub(r'\b(\w+)\s+\1\b', r'\1', text)

    text = re.sub(r'^[^a-z0-9]+', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

def clean_answer(text):
    text = str(text).lower()

    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\b\d{10,}\b', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [5]:
df = df[df.apply(is_valid, axis=1)].copy()

for col in ['question', 'answer', 'intent', 'state', 'crop']:
    df[col] = df[col].fillna('').apply(clean_text)

df = df[df['question'].apply(is_english)]
df = df[df['answer'].apply(is_english)]

df['question'] = df['question'].apply(clean_question)
df['answer'] = df['answer'].apply(clean_answer)


In [6]:
df = df[~df['question'].apply(has_repeat_words)]

df = df[df['question'].str.len() > 10]
df = df[df['answer'].str.len() > 10]

df = df[df['question'].str.split().str.len() > 3]
df = df[df['answer'].str.split().str.len() > 5]

# remove admin answers
df = df[df['answer'].apply(lambda x: not any(k in x for k in [
    'call', 'contact', 'helpline', 'toll free'
]))]

# remove overlong rows
df = df[df['answer'].str.len() < 500]

# prepend crop
df['question'] = df['crop'] + " " + df['question']

# remove frequent useless repeated answers
counts = df['answer'].value_counts()
common_answers = counts[counts > 1000].index
df = df[~df['answer'].isin(common_answers)]

# deduplicate
df = df.drop_duplicates(subset=['question', 'answer'])

# shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("FINAL DATASET SIZE:", len(df))
print(df.sample(5))


FINAL DATASET SIZE: 24888
                                                question  \
22529                      others pm kisan samman yojana   
24681  mango control of leaf cutting weevil using hyd...   
6564   pigeon pea (red gram/arhar/tur) nutrient manag...   
9240   turmeric contact details of turmeric research ...   
8317   pigeon pea (red gram/arhar/tur) nutrient manag...   

                                                  answer  \
22529  registration no. ka245333476 name of farmer re...   
24681  there is no recommendation about control of le...   
6564   suggested that to apply 10:26:26 fertilizer 50...   
9240   turmeric research station, kammarpally : p.sri...   
8317   suggested to apply 10-26-26 at 50 kg per acre ...   

                             intent      state  \
22529            government schemes  karnataka   
24681              plant protection     kerala   
6564            nutrient management  karnataka   
9240   training and exposure visits  telangana   
83

In [7]:

df_final = df[['question', 'answer', 'intent', 'state']]

df_final.to_csv("final_dataset.csv", index=False)

print("Saved final_dataset.csv")

Saved final_dataset.csv


In [8]:
corpus = []

for _, row in df_final.iterrows():
    line = row['question'] + " " + row['answer']
    corpus.append(line)

with open("corpus.txt", "w", encoding="utf-8") as f:
    for line in corpus:
        f.write(line + "\n")

print("Corpus created:", len(corpus))

Corpus created: 24888


In [9]:
def get_vocab(corpus):
    vocab = Counter()

    for line in corpus:
        for word in line.split():
            chars = " ".join(list(word))
            vocab[chars] += 1

    return vocab

vocab = get_vocab(corpus)

print("Initial vocab:", len(vocab))

def get_stats(vocab):
    pairs = Counter()

    for word, freq in vocab.items():
        symbols = word.split()

        for i in range(len(symbols)-1):
            pairs[(symbols[i], symbols[i+1])] += freq

    return pairs

def merge_vocab(pair, vocab):
    new_vocab = {}

    bigram = " ".join(pair)
    replacement = "".join(pair)

    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]

    return new_vocab

num_merges = 3000
merges = []

for i in range(num_merges):
    pairs = get_stats(vocab)

    if not pairs:
        break

    best = max(pairs, key=pairs.get)

    vocab = merge_vocab(best, vocab)
    merges.append(best)

    if i % 500 == 0:
        print("Merge", i, best)

print("Tokenizer training complete")

Initial vocab: 32651
Merge 0 ('a', 'n')
Merge 500 ('ap', 'pli')
Merge 1000 ('b', 'age')
Merge 1500 ('15', 'th')
Merge 2000 ('20', 'ec')
Merge 2500 ('wo', 'und')
Tokenizer training complete


In [10]:
tokens = set()

for word in vocab:
    tokens.update(word.split())

with open("vocab.txt", "w", encoding="utf-8") as f:
    for token in sorted(tokens):
        f.write(token + "\n")

with open("merges.txt", "w", encoding="utf-8") as f:
    for pair in merges:
        f.write(pair[0] + " " + pair[1] + "\n")

print("Saved vocab.txt and merges.txt")

Saved vocab.txt and merges.txt


In [11]:
lengths = [len(x.split()) for x in corpus]

print("Avg Tokens:", sum(lengths)/len(lengths))
print("Max Tokens:", max(lengths))
print("Min Tokens:", min(lengths))


Avg Tokens: 23.95150273224044
Max Tokens: 121
Min Tokens: 11


In [12]:
print("\nDownload Files:\n")

display(FileLink("final_dataset.csv"))
display(FileLink("corpus.txt"))
display(FileLink("vocab.txt"))
display(FileLink("merges.txt"))


Download Files:



/kaggle/working/final_dataset.csv

/kaggle/working/corpus.txt

/kaggle/working/vocab.txt

/kaggle/working/merges.txt